# Phase 3 — NLP: Review Sentiment & Topic Modeling

This notebook covers every step of Phase 3 from start to finish:
- Step 1: Install dependencies & load data sample for validation
- Step 2: Text preprocessing in PySpark (null filter, language detection, cleaning, truncation)
- Step 3: Sentiment scoring with DistilBERT transformer via pandas_udf
- Step 4: Topic modeling via keyword matching + BERTopic exploration
- Step 5: Build and write `gold_nlp_features` Delta table
- Step 6: Scale sentiment to full corpus
- Step 7: Spot-check validation log (5 listings)

**Output table:** `workspace.default.gold_nlp_features`  
**Handed to:** Person 4 (AI/Agent Lead) and Person 5 (Dashboard Lead)

---
## Step 1 — Install Dependencies & Load Data

In [0]:
# Install all required libraries
# Run this cell first, then restart Python before continuing
%pip install transformers torch langdetect bertopic umap-learn hdbscan sentence-transformers
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Verify the gold_reviews table is accessible
spark.table('workspace.default.gold_reviews').printSchema()

root
 |-- listing_id: long (nullable = true)
 |-- city: string (nullable = true)
 |-- review_id: long (nullable = true)
 |-- review_date: date (nullable = true)
 |-- reviewer_id: long (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- comments: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- neighbourhood_cleansed: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- price: double (nullable = true)



In [0]:
spark.table('workspace.default.gold_reviews')

DataFrame[listing_id: bigint, city: string, review_id: bigint, review_date: date, reviewer_id: bigint, reviewer_name: string, comments: string, source_file: string, ingestion_timestamp: timestamp, neighbourhood_cleansed: string, room_type: string, price: double]

In [0]:
%sql
SELECT * 
FROM workspace.default.gold_reviews 
LIMIT 10

listing_id,city,review_id,review_date,reviewer_id,reviewer_name,comments,source_file,ingestion_timestamp,neighbourhood_cleansed,room_type,price
6422,nashville,4435141,2013-05-05,5500449,Jamie,"We both said it.. Michele and her husband are top notch! They kind of spoiled it for all of the other airbnb hosts. :) Their house was beautiful and comfortable. The space was private. They were both very welcoming. Our stay in Nashville was incredible. Highly, Highly, Highly recommend! These two are amazing people!",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 6,private room,43.0
6422,nashville,149883342,2017-05-06,111640667,Jonas,We had a great stay at Michele's place. Due to delayed flights we arrived in the middle of the night but there were still no problem to get a hold of Michele's daughter Keezee so we could get in the house. Nice area close to Shelby Park which is a great place to take a walk and get away from the city life for a few minutes. This house is adorable! I would recommend this to anyone!,nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 6,private room,43.0
6422,nashville,190877177,2017-09-05,112733720,Karen,"Michele's home is quant, comfortable and clean. There is a beautiful park outside her back door that is wonderful for walking, running, biking or just setting and enjoying the beautiful scenery. Michele and her husband Collier were great hosts and made us feel comfortable from the moment we arrived. A great value in a great city.",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 6,private room,43.0
6422,nashville,213065301,2017-11-19,26496163,Ayako,"I had such a pleasant stay at Michele's place. I got there at night, but her house stood out in her neighborhood with a lot of lights. As soon as she opened the door, her warm smile welcomed me. I like her taste of decorating her house. Her house was conveniently located. I just took route #4 bus from downtown. Thank you very much, Michele! I highly recommend her place.",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 6,private room,43.0
39870,nashville,407605798,2019-02-02,239664477,Chris,Great location to the Green Hills area of Nashville. Very clean and easy to find with very good directions.,nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 25,private room,70.0
39870,nashville,443771188,2019-04-26,148233814,Lisa,"Excellent host, very informative, great location, would definitely stay here again & recommend to others if visiting Nashville",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 25,private room,70.0
39870,nashville,1255969445613343573,2024-09-28,237409326,Anna,"Great value, lovely neighborhood and host! Evelyn took us as a last minute reservation and was very kind and accommodating. Would definitely stay again!",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 25,private room,70.0
72906,nashville,53681427,2015-11-11,23429666,Charles,"Richard is a friendly host, and the apartment is clean, well-equipped and in a great location on a quiet street close to several fun neighborhoods (Belmont/Hllsborough Village and 12 South). It's also a short drive or ride to all the downtown activities.",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 18,entire home/apt,91.0
329997,nashville,404900622,2019-01-25,234244457,Viren,A clean feel at home place with all the amenities. And Rick added a personal touch to his service by providing an iron and an ironing board at the very last minute at my request. So thank you Rick. I visit Nashville every few months for work and I will stay again on my next trip at Rick's for sure.,nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 19,entire home/apt,103.0
329997,nashville,900083828078330146,2023-05-26,135892041,Eric,"This was a great space, we really just were here for the night. But did a quick walk downtown. Slept well and had everything we needed! Rick was super responsive and helpful!",nashville_reviews.csv,2026-04-13T02:03:58.118Z,District 19,entire home/apt,103.0


In [0]:
# Load full reviews table and take a ~10K sample for pipeline validation
# CRITICAL: always validate on sample before running on full 3M rows

from pyspark.sql.functions import col, substring

df = spark.table("workspace.default.gold_reviews")

print("Total rows in gold_reviews:", df.count())

sample_df = df.sample(fraction=0.003, seed=42)

print("Sample size:", sample_df.count())

display(
    sample_df
    .select(
        col("listing_id"),
        col("city"),
        substring(col("comments"), 1, 120).alias("sample_comment")
    )
    .limit(5)
)

Total rows in gold_reviews: 2741694
Sample size: 8274


listing_id,city,sample_comment
39870,nashville,Great location to the Green Hills area of Nashville. Very clean and easy to find with very good directions.
12951332,nashville,What a great place!! Everything we needed was on hand. And Elizabeth was a fantastic communicator.
43435358,nashville,Thank you!
44738938,nashville,Pro: Location is very central and easy to get around. Very clean.Con: a bit noisy. Sleep was interrupted by loud m
963846427466202054,nashville,"I had the most wonderful stay here. Sarah was a responsive, kind, and organized host and the AirBnb was spectacular. It"


---
## Step 2 — Text Preprocessing in PySpark

Tasks:
- Filter null comments
- Detect and keep only English reviews using `langdetect`
- Lowercase, remove HTML entities, strip URLs, normalise whitespace
- Truncate to 2000 characters (safely within the 512-token transformer limit)

In [0]:
# Language detection UDF using langdetect

from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
from langdetect import detect, DetectorFactory

# Fix seed for reproducibility
DetectorFactory.seed = 42

def detect_language(text):
    try:
        return detect(text) if text else None
    except:
        return None

lang_udf = udf(detect_language, StringType())

In [0]:
from pyspark.sql.functions import col, lower, regexp_replace, trim, substring

sample_clean = (
    sample_df
    .filter(col("comments").isNotNull())
    .withColumn("language", lang_udf(col("comments")))
    .filter(col("language") == "en")
    .withColumn("clean_comments", lower(col("comments")))
    .withColumn("clean_comments", regexp_replace(col("clean_comments"), r"http\\S+|www\\S+", " "))
    .withColumn("clean_comments", regexp_replace(col("clean_comments"), r"<.*?>", " "))
    .withColumn("clean_comments", regexp_replace(col("clean_comments"), r"&\\w+;", " "))
    .withColumn("clean_comments", regexp_replace(col("clean_comments"), r"\\s+", " "))
    .withColumn("clean_comments", trim(col("clean_comments")))
    .withColumn("clean_comments", substring(col("clean_comments"), 1, 2000))
)

# Avoid double full scans → just show final count
print("After English filter + clean:", sample_clean.count())

display(
    sample_clean
    .select(
        substring(col("comments"), 1, 120).alias("Original"),
        substring(col("clean_comments"), 1, 120).alias("Cleaned"),
        col("language").alias("Lang")
    )
    .limit(5)
)

After English filter + clean: 8036


Original,Cleaned,Lang
Great location to the Green Hills area of Nashville. Very clean and easy to find with very good directions.,great location to the green hills area of nashville. very clean and easy to find with very good directions.,en
What a great place!! Everything we needed was on hand. And Elizabeth was a fantastic communicator.,what a great place!! everything we needed was on hand. and elizabeth was a fantastic communicator.,en
Pro: Location is very central and easy to get around. Very clean.Con: a bit noisy. Sleep was interrupted by loud m,pro: location is very central and easy to get around. very clean. con: a bit noisy. sleep was interrupted by loud music,en
"I had the most wonderful stay here. Sarah was a responsive, kind, and organized host and the AirBnb was spectacular. It","i had the most wonderful stay here. sarah was a responsive, kind, and organized host and the airbnb was spectacular. it",en
The Outlaw Suite was very very cozy and was close to everywhere we went. It was in a private setting/very residential ar,the outlaw suite was very very cozy and was close to everywhere we went. it was in a private setting/very residential ar,en


---
## Step 3 — Sentiment Scoring with DistilBERT Transformer

Model: `distilbert-base-uncased-finetuned-sst-2-english` 

In [0]:
# Quick run on 5 raw reviews before scaling

from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

test_texts = [row["clean_comments"] for row in sample_clean.limit(5).collect()]
test_results = sentiment_model(test_texts, truncation=True)

for t, r in zip(test_texts, test_results):
    print("TEXT:", t[:120])
    print("RESULT:", r)
    print("-" * 60)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

TEXT: great location to the green hills area of nashville.  very clean and easy to find with very good directions.
RESULT: {'label': 'POSITIVE', 'score': 0.9998692274093628}
------------------------------------------------------------
TEXT: what a great place!! everything we needed was on hand. and elizabeth was a fantastic communicator.
RESULT: {'label': 'POSITIVE', 'score': 0.9998639822006226}
------------------------------------------------------------
TEXT: pro: location is very central and easy to get around.  very clean. con: a bit noisy. sleep was interrupted by loud music
RESULT: {'label': 'NEGATIVE', 'score': 0.9951910972595215}
------------------------------------------------------------
TEXT: i had the most wonderful stay here.  sarah was a responsive, kind, and organized host and the airbnb was spectacular. it
RESULT: {'label': 'POSITIVE', 'score': 0.9998390674591064}
------------------------------------------------------------
TEXT: the outlaw suite was very very cozy and 

In [0]:
# Define the pandas_udf for distributed sentiment scoring
# The model is loaded once per executor, not once per row

import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

sentiment_schema = StructType([
    StructField("sentiment_label", StringType(), True),
    StructField("sentiment_score", DoubleType(), True)
])

@pandas_udf(sentiment_schema)
def sentiment_udf(texts: pd.Series) -> pd.DataFrame:
    from transformers import pipeline as hf_pipeline
    model = hf_pipeline(
        "sentiment-analysis",
        model="distilbert-base-uncased-finetuned-sst-2-english",
        truncation=True
    )
    texts_list = texts.fillna("").astype(str).tolist()
    results = model(texts_list, batch_size=8, truncation=True)
    return pd.DataFrame({
        "sentiment_label": [r["label"] for r in results],
        "sentiment_score": [float(r["score"]) for r in results]
    })

In [0]:
# Run sentiment on the validation sample (200 rows → toPandas for speed on Community Edition)
# This validates the full aggregation pipeline before we scale

mini_pdf = (
    sample_clean
    .select("listing_id", "city", "clean_comments")
    .limit(200)
    .toPandas()
)

print("Mini sample shape:", mini_pdf.shape)

# Run model directly on pandas for speed during validation
val_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

val_results = val_model(
    mini_pdf["clean_comments"].fillna("").astype(str).tolist(),
    batch_size=8,
    truncation=True
)

mini_pdf["sentiment_label"] = [r["label"] for r in val_results]
mini_pdf["sentiment_score"] = [float(r["score"]) for r in val_results]

# Signed score: POSITIVE stays positive, NEGATIVE becomes negative
# This gives a -1 to +1 range for intuitive interpretation
mini_pdf["sentiment_signed"] = mini_pdf.apply(
    lambda row: row["sentiment_score"] if row["sentiment_label"] == "POSITIVE" else -row["sentiment_score"],
    axis=1
)

print(mini_pdf[["listing_id", "sentiment_label", "sentiment_score", "sentiment_signed"]].head(10))

Mini sample shape: (200, 3)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

            listing_id sentiment_label  sentiment_score  sentiment_signed
0                39870        POSITIVE         0.999869          0.999869
1             12951332        POSITIVE         0.999864          0.999864
2             44738938        NEGATIVE         0.995191         -0.995191
3   963846427466202054        POSITIVE         0.999839          0.999839
4   995190452009822256        POSITIVE         0.999254          0.999254
5             10353731        POSITIVE         0.999336          0.999336
6             24753664        POSITIVE         0.999731          0.999731
7             35064284        POSITIVE         0.999805          0.999805
8  1085514583905877926        POSITIVE         0.999858          0.999858
9  1391354698549046890        POSITIVE         0.999876          0.999876


In [0]:
# Aggregate sentiment per listing from the validation sample

listing_sentiment_pdf = (
    mini_pdf
    .groupby(["listing_id", "city"], as_index=False)
    .agg(
        avg_sentiment_score=("sentiment_signed", "mean"),
        pct_positive=("sentiment_label", lambda x: round((x == "POSITIVE").mean(), 3)),
        total_reviews=("sentiment_label", "count")
    )
)

print(f"Listings in validation sentiment table: {len(listing_sentiment_pdf)}")
listing_sentiment_pdf.head(10)

Listings in validation sentiment table: 195


,listing_id,city,avg_sentiment_score,pct_positive,total_reviews
0,39870,nashville,0.999869,1.0,1
1,681880,nashville,0.999692,1.0,1
2,2083878,nashville,0.999565,1.0,1
3,3742359,nashville,0.999876,1.0,1
4,3905148,nashville,0.999828,1.0,1
5,4067086,nashville,0.999646,1.0,1
6,4100639,nashville,0.999883,1.0,1
7,4178988,nashville,0.999794,1.0,1
8,4197716,nashville,0.999516,1.0,1
9,4351066,nashville,0.999662,1.0,1


In [0]:
# Convert validation sentiment results back to Spark for joining
listing_sentiment_spark = spark.createDataFrame(listing_sentiment_pdf)
listing_sentiment_spark.show(10, truncate=False)

+----------+---------+-------------------+------------+-------------+
|listing_id|city     |avg_sentiment_score|pct_positive|total_reviews|
+----------+---------+-------------------+------------+-------------+
|39870     |nashville|0.9998692274093628 |1.0         |1            |
|681880    |nashville|0.9996923208236694 |1.0         |1            |
|2083878   |nashville|0.9995649456977844 |1.0         |1            |
|3742359   |nashville|0.9998757839202881 |1.0         |1            |
|3905148   |nashville|0.9998284578323364 |1.0         |1            |
|4067086   |nashville|0.9996459484100342 |1.0         |1            |
|4100639   |nashville|0.9998832941055298 |1.0         |1            |
|4178988   |nashville|0.9997939467430115 |1.0         |1            |
|4197716   |nashville|0.9995162487030029 |1.0         |1            |
|4351066   |nashville|0.9996620416641235 |1.0         |1            |
+----------+---------+-------------------+------------+-------------+
only showing top 10 

---
## Step 4 — Topic Modeling via Keyword Matching

We use keyword-based matching as the primary approach (runs fast on the full corpus in PySpark).  
BERTopic is run separately below as an exploratory supplement.  

Themes tracked: cleanliness, wifi, check-in, location, noise, host responsiveness, value  

**top_praise** = theme with the most total mentions (positive themes: cleanliness, wifi, check-in, location, host, value)  
**top_complaint** = theme with the most mentions from complaint-oriented themes (noise, cleanliness, wifi, checkin, value)

In [0]:
# Keyword matching on the validation sample

from pyspark.sql.functions import when, sum as spark_sum, greatest, lit

theme_df = sample_clean.select("listing_id", "city", "clean_comments")

theme_df = (
    theme_df
    .withColumn(
        "cleanliness_mentions",
        when(col("clean_comments").rlike(r"\b(clean|cleanliness|dirty|spotless|filthy|tidy|stain|dust|mold|mould)\b"), 1).otherwise(0)
    )
    .withColumn(
        "wifi_mentions",
        when(col("clean_comments").rlike(r"\b(wifi|wi-fi|internet|connection|signal|bandwidth|slow wifi)\b"), 1).otherwise(0)
    )
    .withColumn(
        "checkin_mentions",
        when(col("clean_comments").rlike(r"\b(check in|check-in|checkin|arrival|key|lockbox|self check|entry)\b"), 1).otherwise(0)
    )
    .withColumn(
        "location_mentions",
        when(col("clean_comments").rlike(r"\b(location|located|neighborhood|neighbourhood|area|close to|walkable|transit|downtown|central)\b"), 1).otherwise(0)
    )
    .withColumn(
        "noise_mentions",
        when(col("clean_comments").rlike(r"\b(noisy|noise|loud|quiet|sound|sleep|thin walls|street noise)\b"), 1).otherwise(0)
    )
    .withColumn(
        "host_mentions",
        when(col("clean_comments").rlike(r"\b(host|responsive|helpful|communication|communicative|friendly|welcoming|attentive|quick response)\b"), 1).otherwise(0)
    )
    .withColumn(
        "value_mentions",
        when(col("clean_comments").rlike(r"\b(value|worth|price|expensive|affordable|cheap|overpriced|good deal|great deal)\b"), 1).otherwise(0)
    )
)

listing_themes = (
    theme_df
    .groupBy("listing_id", "city")
    .agg(
        spark_sum("cleanliness_mentions").alias("cleanliness_mentions"),
        spark_sum("wifi_mentions").alias("wifi_mentions"),
        spark_sum("checkin_mentions").alias("checkin_mentions"),
        spark_sum("location_mentions").alias("location_mentions"),
        spark_sum("noise_mentions").alias("noise_mentions"),
        spark_sum("host_mentions").alias("host_mentions"),
        spark_sum("value_mentions").alias("value_mentions")
    )
)

display(listing_themes.limit(50).toPandas())

listing_id,city,cleanliness_mentions,wifi_mentions,checkin_mentions,location_mentions,noise_mentions,host_mentions,value_mentions
431258,nashville,0,0,0,0,0,0,0
14010358,seattle,0,0,0,0,0,1,0
49286739,austin,0,0,0,1,1,0,0
30820481,chicago,0,0,0,1,0,1,0
10107241,twin_cities,0,0,0,1,0,0,0
30346059,nashville,2,0,2,2,1,0,0
840730480966735071,chicago,0,0,1,1,0,1,0
8049698,nashville,0,0,0,0,0,1,0
1016102760719961694,seattle,1,0,0,1,0,0,0
46278553,chicago,1,0,0,1,1,0,0


---
## Step 5 — Build the Validation gold_nlp_features Table

Joins sentiment + themes, derives `top_praise` and `top_complaint`, adds `sentiment_category`.  
This version is built from the validation sample. The full-corpus version is in Step 6.

In [0]:
# Helper function for top_praise: pick the theme column with the highest count
# Covers all 6 positive-leaning themes

from pyspark.sql.functions import round as spark_round

def add_top_praise(df):
    praise_cols = ["cleanliness_mentions", "wifi_mentions", "checkin_mentions",
                   "location_mentions", "host_mentions", "value_mentions"]
    praise_labels = ["cleanliness", "wifi", "check-in", "location", "host responsiveness", "value"]
    praise_max = greatest(*[col(c) for c in praise_cols])
    expr = None
    for c, label in zip(praise_cols, praise_labels):
        cond = when(praise_max == col(c), lit(label))
        expr = cond if expr is None else expr.when(praise_max == col(c), lit(label))
    return df.withColumn("top_praise", expr.otherwise(lit("none")))

def add_top_complaint(df):
    # Complaint candidates: noise, cleanliness, wifi, value, check-in (all can be pain points)
    complaint_cols = ["noise_mentions", "cleanliness_mentions", "wifi_mentions",
                      "value_mentions", "checkin_mentions"]
    complaint_labels = ["noise", "cleanliness", "wifi", "value", "check-in"]
    complaint_max = greatest(*[col(c) for c in complaint_cols])
    expr = None
    for c, label in zip(complaint_cols, complaint_labels):
        cond = when(complaint_max == col(c), lit(label))
        expr = cond if expr is None else expr.when(complaint_max == col(c), lit(label))
    return df.withColumn("top_complaint", expr.otherwise(lit("none")))

def add_sentiment_category(df):
    return df.withColumn(
        "sentiment_category",
        when(col("avg_sentiment_score") > 0.5,  "Very Positive")
        .when(col("avg_sentiment_score") > 0.2,  "Positive")
        .when(col("avg_sentiment_score") > -0.2, "Neutral")
        .when(col("avg_sentiment_score") > -0.5, "Negative")
        .otherwise("Very Negative")
    )

In [0]:
# Build the sample gold_nlp_features (for validation)

gold_nlp_sample = (
    listing_sentiment_spark
    .join(listing_themes, on=["listing_id", "city"], how="inner")
)

gold_nlp_sample = add_sentiment_category(gold_nlp_sample)
gold_nlp_sample = add_top_praise(gold_nlp_sample)
gold_nlp_sample = add_top_complaint(gold_nlp_sample)

gold_nlp_sample = (
    gold_nlp_sample
    .withColumn("avg_sentiment_score", spark_round(col("avg_sentiment_score"), 3))
    .withColumn("pct_positive",        spark_round(col("pct_positive"), 3))
)

# Select final column order matching the spec
gold_nlp_sample = gold_nlp_sample.select(
    "listing_id",
    "city",
    "avg_sentiment_score",
    "sentiment_category",
    "pct_positive",
    "total_reviews",
    "top_complaint",
    "top_praise",
    "cleanliness_mentions",
    "wifi_mentions",
    "checkin_mentions",
    "location_mentions",
    "noise_mentions",
    "host_mentions",
    "value_mentions"
)

display(gold_nlp_sample.limit(50).toPandas())

listing_id,city,avg_sentiment_score,sentiment_category,pct_positive,total_reviews,top_complaint,top_praise,cleanliness_mentions,wifi_mentions,checkin_mentions,location_mentions,noise_mentions,host_mentions,value_mentions
8049698,nashville,1.0,Very Positive,1.0,1,noise,host responsiveness,0,0,0,0,0,1,0
37049775,nashville,1.0,Very Positive,1.0,1,cleanliness,cleanliness,1,0,0,0,0,0,0
35038778,nashville,1.0,Very Positive,1.0,1,check-in,check-in,0,0,1,0,0,0,0
33745251,nashville,0.997,Very Positive,1.0,1,value,location,0,0,0,2,0,0,1
11375711,nashville,1.0,Very Positive,1.0,1,noise,location,0,0,0,1,0,0,0
963819899476246520,nashville,0.999,Very Positive,1.0,1,noise,location,0,0,0,1,0,0,0
23169939,nashville,1.0,Very Positive,1.0,1,cleanliness,cleanliness,2,0,0,1,0,1,0
1181095796850199916,nashville,1.0,Very Positive,1.0,1,noise,cleanliness,0,0,0,0,0,0,0
7329327,nashville,1.0,Very Positive,1.0,1,noise,cleanliness,0,0,0,0,0,0,0
14389733,nashville,1.0,Very Positive,1.0,1,noise,host responsiveness,1,0,1,1,2,2,1


---
## Step 6 — Scale to Full Corpus & Write gold_nlp_features to Delta

Now that the pipeline is validated on the sample, we run it on the full `gold_reviews` table.  
Sentiment is computed by converting a larger pandas batch; themes are computed fully in PySpark Spark.

In [0]:
# COMPLETE STEP 6: a,b,c,d — Run this one cell only
# Step 6a — Reload and clean full reviews table
# b → listing_themes_full — keyword mention counts (cleanliness, wifi, checkin, location, noise, host, value) grouped by listing_id and city
# c — Sentiment via lexicon scoring (no model download)
# VADER is built for short informal text like reviews — installs in seconds
# d → gold_nlp_features — the final table, which is the join of sentiment + themes, with sentiment_category, top_praise, and top_complaint added on top

import subprocess
subprocess.run(["pip", "install", "vaderSentiment", "-q"])

from pyspark.sql.functions import col, lower, regexp_replace, trim, substring, when, greatest, lit, sum as spark_sum, round as spark_round
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

# ── full_clean ────────────────────────────────────────────────────
full_reviews = spark.table("workspace.default.gold_reviews")

full_clean = (
    full_reviews
    .filter(col("comments").isNotNull())
    .withColumn("clean_comments", lower(col("comments")))
    .withColumn("clean_comments", regexp_replace(col("clean_comments"), r"http\S+|www\S+", " "))
    .withColumn("clean_comments", regexp_replace(col("clean_comments"), r"<.*?>", " "))
    .withColumn("clean_comments", regexp_replace(col("clean_comments"), r"&\w+;", " "))
    .withColumn("clean_comments", regexp_replace(col("clean_comments"), r"\s+", " "))
    .withColumn("clean_comments", trim(col("clean_comments")))
    .withColumn("clean_comments", substring(col("clean_comments"), 1, 2000))
)
print(" full_clean ready")

# ── listing_themes_full ───────────────────────────────────────────
listing_themes_full = (
    full_clean
    .select("listing_id", "city", "clean_comments")
    .withColumn("cleanliness_mentions", when(col("clean_comments").rlike(r"\b(clean|cleanliness|dirty|spotless|filthy|tidy|stain|dust|mold|mould)\b"), 1).otherwise(0))
    .withColumn("wifi_mentions",        when(col("clean_comments").rlike(r"\b(wifi|wi-fi|internet|connection|signal|bandwidth)\b"), 1).otherwise(0))
    .withColumn("checkin_mentions",     when(col("clean_comments").rlike(r"\b(check in|check-in|checkin|arrival|key|lockbox|self check|entry)\b"), 1).otherwise(0))
    .withColumn("location_mentions",   when(col("clean_comments").rlike(r"\b(location|located|neighborhood|neighbourhood|area|close to|walkable|transit|downtown|central)\b"), 1).otherwise(0))
    .withColumn("noise_mentions",       when(col("clean_comments").rlike(r"\b(noisy|noise|loud|quiet|sound|sleep|thin walls|street noise)\b"), 1).otherwise(0))
    .withColumn("host_mentions",        when(col("clean_comments").rlike(r"\b(host|responsive|helpful|communication|communicative|friendly|welcoming|attentive)\b"), 1).otherwise(0))
    .withColumn("value_mentions",       when(col("clean_comments").rlike(r"\b(value|worth|price|expensive|affordable|cheap|overpriced|good deal|great deal)\b"), 1).otherwise(0))
    .groupBy("listing_id", "city")
    .agg(
        spark_sum("cleanliness_mentions").alias("cleanliness_mentions"),
        spark_sum("wifi_mentions").alias("wifi_mentions"),
        spark_sum("checkin_mentions").alias("checkin_mentions"),
        spark_sum("location_mentions").alias("location_mentions"),
        spark_sum("noise_mentions").alias("noise_mentions"),
        spark_sum("host_mentions").alias("host_mentions"),
        spark_sum("value_mentions").alias("value_mentions")
    )
)
print(" listing_themes_full ready")

# ── full_listing_sentiment ────────────────────────────────────────
analyzer = SentimentIntensityAnalyzer()

def score_text(text):
    scores = analyzer.polarity_scores(str(text))
    compound = scores["compound"]
    label = "POSITIVE" if compound >= 0.05 else "NEGATIVE" if compound <= -0.05 else "NEUTRAL"
    return compound, label

cities = [row["city"] for row in full_clean.select("city").distinct().collect()]
print(f"Cities found: {cities}")

all_results = []
for city in cities:
    city_pdf = (
        full_clean
        .filter(col("city") == city)
        .select("listing_id", "city", "clean_comments")
        .toPandas()
    )
    city_pdf[["sentiment_signed", "sentiment_label"]] = city_pdf["clean_comments"].apply(
        lambda t: score_text(t)
    ).apply(lambda x: pd.Series(x))
    all_results.append(city_pdf)
    print(f"  ✓ {city}: {len(city_pdf)} rows scored")

sentiment_pdf = pd.concat(all_results, ignore_index=True)

full_listing_sentiment = spark.createDataFrame(
    sentiment_pdf
    .groupby(["listing_id", "city"], as_index=False)
    .agg(
        avg_sentiment_score=("sentiment_signed", "mean"),
        pct_positive=("sentiment_label", lambda x: round((x == "POSITIVE").mean(), 3)),
        total_reviews=("sentiment_label", "count")
    )
)
print(f" full_listing_sentiment ready. Listings: {full_listing_sentiment.count()}")

# ── Helper functions ──────────────────────────────────────────────
def add_sentiment_category(df):
    return df.withColumn(
        "sentiment_category",
        when(col("avg_sentiment_score") > 0.5,  "Very Positive")
        .when(col("avg_sentiment_score") > 0.2,  "Positive")
        .when(col("avg_sentiment_score") > -0.2, "Neutral")
        .when(col("avg_sentiment_score") > -0.5, "Negative")
        .otherwise("Very Negative")
    )

def add_top_praise(df):
    praise_cols   = ["cleanliness_mentions", "wifi_mentions", "checkin_mentions",
                     "location_mentions", "host_mentions", "value_mentions"]
    praise_labels = ["cleanliness", "wifi", "check-in", "location", "host responsiveness", "value"]
    praise_max = greatest(*[col(c) for c in praise_cols])
    expr = None
    for c, label in zip(praise_cols, praise_labels):
        cond = when(praise_max == col(c), lit(label))
        expr = cond if expr is None else expr.when(praise_max == col(c), lit(label))
    return df.withColumn("top_praise", expr.otherwise(lit("none")))

def add_top_complaint(df):
    complaint_cols   = ["noise_mentions", "cleanliness_mentions", "wifi_mentions",
                        "value_mentions", "checkin_mentions"]
    complaint_labels = ["noise", "cleanliness", "wifi", "value", "check-in"]
    complaint_max = greatest(*[col(c) for c in complaint_cols])
    expr = None
    for c, label in zip(complaint_cols, complaint_labels):
        cond = when(complaint_max == col(c), lit(label))
        expr = cond if expr is None else expr.when(complaint_max == col(c), lit(label))
    return df.withColumn("top_complaint", expr.otherwise(lit("none")))

# ── Final join ────────────────────────────────────────────────────
gold_nlp_features = (
    full_listing_sentiment
    .join(listing_themes_full, on=["listing_id", "city"], how="inner")
)

gold_nlp_features = add_sentiment_category(gold_nlp_features)
gold_nlp_features = add_top_praise(gold_nlp_features)
gold_nlp_features = add_top_complaint(gold_nlp_features)

gold_nlp_features = (
    gold_nlp_features
    .withColumn("avg_sentiment_score", spark_round(col("avg_sentiment_score"), 3))
    .withColumn("pct_positive",        spark_round(col("pct_positive"), 3))
    .select(
        "listing_id", "city", "avg_sentiment_score", "sentiment_category",
        "pct_positive", "total_reviews", "top_complaint", "top_praise",
        "cleanliness_mentions", "wifi_mentions", "checkin_mentions",
        "location_mentions", "noise_mentions", "host_mentions", "value_mentions"
    )
)

print("\n gold_nlp_features ready")
print("Row count:", gold_nlp_features.count())
print("\nCity breakdown:")
gold_nlp_features.groupBy("city").count().orderBy("count", ascending=False).show(50, truncate=False)

display(gold_nlp_features.limit(100))


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


 full_clean ready
 listing_themes_full ready
Cities found: ['austin', 'nashville', 'seattle', 'chicago', 'twin_cities']
  ✓ austin: 588211 rows scored
  ✓ nashville: 784756 rows scored
  ✓ seattle: 575718 rows scored
  ✓ chicago: 492377 rows scored
  ✓ twin_cities: 300632 rows scored
 full_listing_sentiment ready. Listings: 35217

 gold_nlp_features ready
Row count: 35217

City breakdown:
+-----------+-----+
|city       |count|
+-----------+-----+
|austin     |8909 |
|nashville  |8449 |
|chicago    |6977 |
|seattle    |6109 |
|twin_cities|4773 |
+-----------+-----+



listing_id,city,avg_sentiment_score,sentiment_category,pct_positive,total_reviews,top_complaint,top_praise,cleanliness_mentions,wifi_mentions,checkin_mentions,location_mentions,noise_mentions,host_mentions,value_mentions
23872834,nashville,0.808,Very Positive,0.976,417,cleanliness,location,138,2,35,258,42,84,22
878440666274505872,nashville,0.757,Very Positive,0.939,197,cleanliness,location,29,1,6,95,10,41,10
1085053437088691265,nashville,0.889,Very Positive,1.0,2,cleanliness,cleanliness,2,0,0,1,0,1,0
603642530303706197,nashville,0.818,Very Positive,0.965,114,cleanliness,location,31,0,5,42,4,33,9
1261212592859308250,nashville,0.87,Very Positive,1.0,45,cleanliness,location,14,0,3,23,5,16,0
1059942070793056811,nashville,0.841,Very Positive,0.974,77,cleanliness,location,25,1,5,51,4,37,4
1313140860879428198,nashville,0.739,Very Positive,0.926,27,cleanliness,location,9,0,4,13,3,11,1
48068437,nashville,0.808,Very Positive,0.979,47,cleanliness,location,13,0,2,29,2,8,4
1021950121967174724,nashville,0.811,Very Positive,0.97,203,cleanliness,location,48,0,9,123,12,78,9
48396286,nashville,0.846,Very Positive,1.0,22,cleanliness,cleanliness,8,0,4,6,1,4,0


In [0]:
gold_nlp_features.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_nlp_features")

verify = spark.table("gold_nlp_features")
print("Row count in Delta table:", verify.count())
display(verify.limit(50))


Row count in Delta table: 35217


listing_id,city,avg_sentiment_score,pct_positive,total_reviews,cleanliness_mentions,wifi_mentions,checkin_mentions,location_mentions,noise_mentions,host_mentions,value_mentions,sentiment_category,top_praise,top_complaint
23872834,nashville,0.808,0.976,417,138,2,35,258,42,84,22,Very Positive,location,cleanliness
878440666274505872,nashville,0.757,0.939,197,29,1,6,95,10,41,10,Very Positive,location,cleanliness
1085053437088691265,nashville,0.889,1.0,2,2,0,0,1,0,1,0,Very Positive,cleanliness,cleanliness
603642530303706197,nashville,0.818,0.965,114,31,0,5,42,4,33,9,Very Positive,location,cleanliness
1261212592859308250,nashville,0.87,1.0,45,14,0,3,23,5,16,0,Very Positive,location,cleanliness
1059942070793056811,nashville,0.841,0.974,77,25,1,5,51,4,37,4,Very Positive,location,cleanliness
1313140860879428198,nashville,0.739,0.926,27,9,0,4,13,3,11,1,Very Positive,location,cleanliness
48068437,nashville,0.808,0.979,47,13,0,2,29,2,8,4,Very Positive,location,cleanliness
1021950121967174724,nashville,0.811,0.97,203,48,0,9,123,12,78,9,Very Positive,location,cleanliness
48396286,nashville,0.846,1.0,22,8,0,4,6,1,4,0,Very Positive,cleanliness,cleanliness


---
## Step 7 — Spot-Check Validation Log

Manually verify 5 listings: read their raw reviews, check that the sentiment score and top themes make sense.  
This is a graded deliverable — the log below is required for the GitHub README (handed to Person 5).

In [0]:
# Pick 5 listing IDs from the output table for manual spot-check

spot_check_ids = (
    spark.table("workspace.default.gold_nlp_features")
    .select("listing_id", "city")
    .limit(5)
    .collect()
)

print("Spot-check listing IDs:")
for row in spot_check_ids:
    print(f"  listing_id={row['listing_id']}, city={row['city']}")

Spot-check listing IDs:
  listing_id=23872834, city=nashville
  listing_id=878440666274505872, city=nashville
  listing_id=1085053437088691265, city=nashville
  listing_id=603642530303706197, city=nashville
  listing_id=1261212592859308250, city=nashville


In [0]:
from pyspark.sql.functions import col

nlp_table = spark.table("workspace.default.gold_nlp_features")
reviews_table = spark.table("workspace.default.gold_reviews")

for row in spot_check_ids:
    lid = row["listing_id"]
    city = row["city"]

    print("=" * 70)
    print(f"LISTING: {lid} | CITY: {city}")
    print()

    print("NLP Features")
    display(
        nlp_table
        .filter((col("listing_id") == lid) & (col("city") == city))
        .select(
            "listing_id",
            "city",
            "avg_sentiment_score",
            "sentiment_category",
            "pct_positive",
            "total_reviews",
            "top_praise",
            "top_complaint"
        )
    )

    print("Sample Reviews")
    display(
        reviews_table
        .filter((col("listing_id") == lid) & (col("city") == city))
        .select(
            "listing_id",
            "city",
            col("comments").alias("sample_review")
        )
        .limit(3)
    )

LISTING: 23872834 | CITY: nashville

NLP Features


listing_id,city,avg_sentiment_score,sentiment_category,pct_positive,total_reviews,top_praise,top_complaint
23872834,nashville,0.808,Very Positive,0.976,417,location,cleanliness


Sample Reviews


listing_id,city,sample_review
23872834,nashville,"The place provided what we needed when traveling with another couple, 2 KING size beds & 2 baths as well as being a relatively close walk to Broadway. We did not use the kitchen other than making coffee in morning and it was nice to have a Keurig and starter K-cups provided. There’s also a coffee shop around the corner. I do recommend putting down a bath rug in front of toilet & sink as tile flooring in master bath was very cold. All in all we would probably stay again if we had another couple with us. Thank you."
23872834,nashville,This was the Best Airbnb I've ever stayed at. We loved Nashville and will definitely stay here again.
23872834,nashville,"Wonderful location, just a few blocks away from everything. We look forward to visiting again. We got food from Till 5 Pizza just around the corner and highly recommend them, large menu and very convenient. Property was nicely decorated, clean and stocked with kuerig coffee, thank you so much for everything!"


LISTING: 878440666274505872 | CITY: nashville

NLP Features


...

listing_id,city,sample_review
603642530303706197,nashville,"This place is adorable, decor is super cute! It has lots of space, and very convenient to all attractions. The floors could use a good cleaning, but over all was clean. The neighborhood is a little sketchy, it seems like an older neighborhood that they are revitalizing. However, we didn’t have any problems, the garage was a huge plus to park the car overnight. It was definitely a cute place to stay that was comfortable and enjoyable! Can’t wait to come back to Nashville!"
603642530303706197,nashville,A remarkable place to stay! We enjoyed.
603642530303706197,nashville,We had a blast! The house is everything we expected didn’t have any issues


LISTING: 1261212592859308250 | CITY: nashville

NLP Features


listing_id,city,avg_sentiment_score,sentiment_category,pct_positive,total_reviews,top_praise,top_complaint
1261212592859308250,nashville,0.87,Very Positive,1.0,45,location,cleanliness


Sample Reviews


listing_id,city,sample_review
1261212592859308250,nashville,"Our stay at this Airbnb was fantastic! The location was ideal, offering easy access to downtown Broadway while still providing a peaceful retreat. The home itself was impeccably clean and well-maintained, making for a truly comfortable and enjoyable experience."
1261212592859308250,nashville,Perfect space for a group of 8 guys. Plenty of space and seating available for all of us. The place was very clean and had all of the essentials. Great location with it being only a 10 minute drive to downtown. 10/10 will be back in this Airbnb for our next visit.
1261212592859308250,nashville,"This place is magnificent. It is definitely a friendly party-house. The rooms are comfortable however, if you are not comfortable with stairs, this is not your place. The rooftop patio was awesome and the living room sofa was comfortable. The transformer table was very impressive for our sitdown family dinner for our last night. My friend said the middle floor shower had the best water pressure however, I thought mine was just fine. This house is perfect for a long weekend. There is a ground floor bedroom that has two or three steps to get to. There is one bedroom on the kitchen/living area level and another two bedrooms on the floor above this. The rooftop patio is another flight of stairs on top of that. Something that might be helpful is a label for the washer and dryer. My friend had never used an up/down and was trying to wash her clothes in the dryer. Luckily I came to add my clothing in and switched. But it was hard to distinguish which was which. Marvelous place. We loved it!"


---
## (Supplemental) BERTopic Exploration

BERTopic is run here on a 2,000-row sample for exploratory topic discovery.  
Results informed the keyword list used in the primary pipeline above.  
This section is NOT required to run for the Delta table output — it is supplemental.

In [0]:
# BERTopic on 2,000-row sample for topic exploration

from pyspark.sql.functions import col, lower, regexp_replace, trim

bert_df = spark.table("workspace.default.gold_reviews")

bert_clean = (
    bert_df
    .filter(col("comments").isNotNull())
    .withColumn("clean_comments", lower(col("comments")))
    .withColumn("clean_comments", regexp_replace(col("clean_comments"), r"http\S+|www\S+", " "))
    .withColumn("clean_comments", regexp_replace(col("clean_comments"), r"<.*?>", " "))
    .withColumn("clean_comments", regexp_replace(col("clean_comments"), r"&\w+;", " "))
    .withColumn("clean_comments", regexp_replace(col("clean_comments"), r"\s+", " "))
    .withColumn("clean_comments", trim(col("clean_comments")))
)

topic_pdf = (
    bert_clean
    .select("listing_id", "city", "clean_comments")
    .filter(col("clean_comments").isNotNull())
    .limit(2000)
    .toPandas()
)

topic_pdf = topic_pdf.dropna(subset=["clean_comments"]).copy()
topic_pdf["clean_comments"] = topic_pdf["clean_comments"].astype(str).str.strip()
topic_pdf = topic_pdf[topic_pdf["clean_comments"] != ""]

print("BERTopic input shape:", topic_pdf.shape)

BERTopic input shape: (2000, 3)


In [0]:
from bertopic import BERTopic

topic_model = BERTopic(
    language="english",
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model.fit_transform(topic_pdf["clean_comments"].tolist())
topic_pdf["topic"] = topics

topic_info = topic_model.get_topic_info()
print("Top 20 discovered topics:")
display(topic_info.head(20))

2026-04-20 03:01:31,734 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

2026-04-20 03:02:16,198 - BERTopic - Embedding - Completed ✓
2026-04-20 03:02:16,200 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-20 03:02:30,676 - BERTopic - Dimensionality - Completed ✓
2026-04-20 03:02:30,677 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-20 03:02:30,728 - BERTopic - Cluster - Completed ✓
2026-04-20 03:02:30,733 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-20 03:02:30,804 - BERTopic - Representation - Completed ✓


Top 20 discovered topics:


Topic Count Name Representation Representative_Docs -1 744 -1_and_the_was_to List(and, the, was, to, we, great, very, for, stay, place) List(first on allison, she was wonderful. from beginning to end, very responsive. she recommended exactly what i was looking for when i wasn't sure. unfortunately i wasn't able to meet her but i felt she was just as much there with her speedy responses. on the place, it was great. felt a bit secluded which was great for me when i retired from the downtown hustle and bustle. the place was very modern, clean, sort of minimalistic. love the kitchen area. the bedroom was what you needed. very quaint. only wish there were hangers in the closet. the only thing you may should know, is the house is very airy so it's naturally chilly. but the thermostat works pretty well and have an emergency heat in case you need it to warm up quick. recommend this place to anyone! would not mind coming back at all. plus she really knows her areas and that was a green light for me because i was inquiring to move there. she was the perfect person for this stay!, we could not have asked for a better place for a girls weekend. when we walked in, every one us commented on just how perfect it was. the house is roomy and well furnished. we made use of the wine and champagne glasses and had everything we needed to cook breakfast. we did not use the outdoor space because of the weather, but it looked inviting. there are several shops within walking distance. when we were ready to go downtown, it was just a short uber ride away. my only wish is that we had stayed longer. if i am ever back in nashville, i will definitely try to stay in this adorable house again., margaret’s place was super adorable and very clean! the location was perfect, and a quick uber ride to broadway street! there are tons of beds and three bathrooms so it’s great for lots of people. we just had a few issues so i couldn’t give 5 stars. we arrived and the driveway and the walkway were completely full of snow. i had texted margaret about it and she responded quickly and said she was unable to get to the property to shovel as she was also stuck at her house due to the weather. i completely understood but it was still very inconvenient to unload our things. luckily we received the code to the front door because we couldn’t even get the back gate open, because of all of the snow, to go in the back way. the hot tub had an odor and was not very warm. we did have to reset it, which we found in her rule book, and then had to shock it which helped in the end. also, just inconvenient. the water pressure in the one shower wasn’t great. in the other shower, the shower head kept falling off and almost hit my mom in the head. this house is definitely made for the spring and summer! the back patio area is great, we just couldn’t use it much because of the snow. overall we had a great experience. margaret was super friendly and easy to get ahold of. we just had some small inconveniences during our stay. but we would stay here again! the location and back patio are wonderful! :)) 0 422 0_nashville_to_and_in List(nashville, to, and, in, the, we, was, stay, for, is) List(we loved the place! close to everything, place was clean and super cute. definitely will stay here again if we come back to nashville!, we had a wonderful stay at this beautiful townhouse in nashville! the space was clean, comfortable, and stylish — perfect for a relaxing getaway. the location was super convenient, making it easy to explore the city, and the neighborhood felt safe and quiet. the beds were cozy, and we had everything we needed during our stay. our host was friendly, responsive, and made the whole process seamless. we really appreciated the attention to detail and felt right at home. highly recommend this place if you’re visiting nashville — we’d definitely stay here again!, we loved staying here! it was a beautiful space that was very clean and matched all the photos! it was just the two of u

In [0]:
# Print top keywords per topic to understand themes

print("Top keywords per topic:\n")
for topic_id in topic_info["Topic"].head(15):
    if topic_id != -1:
        keywords = topic_model.get_topic(topic_id)
        kw_str = ", ".join([w for w, _ in keywords[:8]])
        print(f"Topic {topic_id:>3}: {kw_str}")

Top keywords per topic:

Topic   0: nashville, to, and, in, the, we, was, stay
Topic   1: broadway, to, and, the, was, great, very, of
Topic   2: clean, place, great, very, stay, and, nice, to
Topic   3: house, and, home, the, we, was, for, very
Topic   4: host, great, responsive, and, very, hosts, was, nice
Topic   5: airbnb, the, is, and, to, of, in, was
Topic   6: location, great, place, perfect, and, value, easy, close
Topic   7: airbnb, nashville, this, was, and, the, to, we
Topic   8: the, we, tv, and, was, were, to, had
Topic   9: de, muy, un, et, trs, en, por, casa
Topic  10: place, stay, great, to, nice, staying, will, lee
Topic  11: again, stay, would, place, definitely, will, great, wait
Topic  12: loft, and, the, is, was, to, beautiful, very
Topic  13: space, of, the, for, and, heart, in, group


In [0]:
# Check which cities are in the output
spark.sql("""
    SELECT city, COUNT(*) as listing_count
    FROM workspace.default.gold_nlp_features
    GROUP BY city
    ORDER BY listing_count DESC
""").show(50, truncate=False)

+-----------+-------------+
|city       |listing_count|
+-----------+-------------+
|austin     |8909         |
|nashville  |8449         |
|chicago    |6977         |
|seattle    |6109         |
|twin_cities|4773         |
+-----------+-------------+



In [0]:
from pyspark.sql.functions import col

nlp_table = spark.table("workspace.default.gold_nlp_features")

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|       gold_calendar|      false|
| default| gold_calendar_daily|      false|
| default|       gold_features|      false|
| default|       gold_listings|      false|
| default|   gold_nlp_features|      false|
| default|gold_nlp_features...|      false|
| default|gold_nlp_features...|      false|
| default|        gold_reviews|      false|
+--------+--------------------+-----------+



In [0]:
nlp_table = spark.table("default.gold_nlp_features")

display(nlp_table.limit(5))  # optional check

listing_id,city,avg_sentiment_score,pct_positive,total_reviews,cleanliness_mentions,wifi_mentions,checkin_mentions,location_mentions,noise_mentions,host_mentions,value_mentions,sentiment_category,top_praise,top_complaint
23872834,nashville,0.808,0.976,417,138,2,35,258,42,84,22,Very Positive,location,cleanliness
878440666274505872,nashville,0.757,0.939,197,29,1,6,95,10,41,10,Very Positive,location,cleanliness
1085053437088691265,nashville,0.889,1.0,2,2,0,0,1,0,1,0,Very Positive,cleanliness,cleanliness
603642530303706197,nashville,0.818,0.965,114,31,0,5,42,4,33,9,Very Positive,location,cleanliness
1261212592859308250,nashville,0.87,1.0,45,14,0,3,23,5,16,0,Very Positive,location,cleanliness


In [0]:
nlp_table.write \
    .mode("overwrite") \
    .saveAsTable("default.gold_nlp_features_parquet")

In [0]:
check_table = spark.table("default.gold_nlp_features_parquet")
display(check_table.limit(5))
print(check_table.count())

listing_id,city,avg_sentiment_score,pct_positive,total_reviews,cleanliness_mentions,wifi_mentions,checkin_mentions,location_mentions,noise_mentions,host_mentions,value_mentions,sentiment_category,top_praise,top_complaint
23872834,nashville,0.808,0.976,417,138,2,35,258,42,84,22,Very Positive,location,cleanliness
878440666274505872,nashville,0.757,0.939,197,29,1,6,95,10,41,10,Very Positive,location,cleanliness
1085053437088691265,nashville,0.889,1.0,2,2,0,0,1,0,1,0,Very Positive,cleanliness,cleanliness
603642530303706197,nashville,0.818,0.965,114,31,0,5,42,4,33,9,Very Positive,location,cleanliness
1261212592859308250,nashville,0.87,1.0,45,14,0,3,23,5,16,0,Very Positive,location,cleanliness


35217


In [0]:
nlp_table.write.mode("overwrite").saveAsTable("default.gold_nlp_features_shared")

In [0]:
nlp_table = spark.table("default.gold_nlp_features")

nlp_table.write \
    .mode("overwrite") \
    .saveAsTable("default.gold_nlp_features_shared")

print(f"Done. Rows: {nlp_table.count()}")

Done. Rows: 35217


In [0]:

gold_nlp_features.createOrReplaceTempView("gold_nlp_temp")
print(f"Temp view created. Rows: {gold_nlp_features.count()}")

Temp view created. Rows: 35217
